# 第 1 周末练习 —— 系统设计面试教练

## 练习目标（理念）

官方周末题要求做一个「技术问答 + 解释」小工具；本笔记本把同一套 API 熟练度用到**系统设计面试练习**上：

- **输入**：你的简历文本 + 出题提示
- **中间产物**：一道贴合背景的系统设计题
- **输出**：对你作答的打分与结构化反馈（Grade / What went well / Gaps / Better answer outline）

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `openai.chat.completions.create(...)` |
| `messages`（system / user） | system 定面试官人设；user 塞简历、题目、作答 |
| 模型名常量 | `MODEL_GPT`（此处实际指向本地 Ollama 的 `phi3`） |
| OpenAI 兼容本地接口 | `base_url=http://localhost:11434/v1`，密钥占位 `"ollama"` |
| 笔记本展示 | `display(Markdown(...))` 渲染题目与反馈 |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. 本机需已启动 Ollama，并 `ollama pull phi3`（或把 `MODEL_GPT` 改回云端 `gpt-4o-mini` 并改用真实 `OPENAI_API_KEY`）
3. 在简历单元格把 `<PASTE_YOUR_RESUME_HERE>` 换成你的简历；跑生成题 → 填写作答 → 再跑评分格


In [17]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：在笔记本里漂亮地显示 Markdown
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类：既可打云端，也可打 Ollama 的 OpenAI 兼容接口
from openai import OpenAI


In [ ]:
# ========== 常量：模型名字集中写在一处，后面只改这里 ==========

# 云端备选（当前注释掉）：若改回 OpenAI 云端，取消下一行注释并改客户端 base_url / api_key
# MODEL_GPT = "gpt-4o-mini" # OpenAI
# 本地 Ollama 模型名：字符串必须和本机已安装的模型名一致（需事先 ollama pull phi3）
MODEL_GPT = "phi3" # Ollama


In [ ]:
# ========== 环境 + 客户端：走本地 Ollama 的 OpenAI 兼容 /v1 ==========

# 加载 .env：override=True 表示 .env 里的值覆盖进程里已有同名环境变量
load_dotenv(override=True)
# 从环境读取 OPENAI_API_KEY（本笔记本虽接 Ollama，仍沿用课程常见的密钥检查习惯）
api_key = os.getenv("OPENAI_API_KEY")

# 没有密钥就立刻失败，避免后面请求才报含糊错误
if not api_key:
    # 错误文案保持英文原文：影响排查/依赖判断的字符串不翻译
    raise ValueError("No OPENAI_API_KEY found. Add it to your .env file.")

# 创建 OpenAI 客户端，但指向本机 Ollama：base_url 的 /v1 是 OpenAI 兼容层
openai = OpenAI(
    base_url="http://localhost:11434/v1",  # Local Ollama API
    # Ollama 兼容接口通常要求任意非空 api_key；这里用占位字符串 "ollama"
    api_key="ollama"
)


## 粘贴你的简历 + 出题提示

下一格把简历原文放进 `resume_text`，并用 `user_prompt` 告诉模型「出什么样的题」。  
发给模型的 prompt 字符串保持英文（可运行 / 影响回答风格，不翻译）。


In [ ]:
# ========== 输入：简历文本 + 用户侧出题提示 ==========

# 在下面粘贴您的简历文本（占位符换成真实内容后再跑生成题单元格）
resume_text = """
<PASTE_YOUR_RESUME_HERE>
"""

# user_prompt：告诉模型如何出题；保留英文，避免改变出题风格
user_prompt = """
Create one realistic system design interview use-case question based on my resume.
Keep it challenging but aligned with my background.
"""


In [27]:
# ========== 调用 Chat Completions：根据简历生成一道系统设计题 ==========

# messages：对话列表；system 定「面试官」角色，user 提供简历与出题要求
question_messages = [
    {
        "role": "system",
        # system prompt 保留英文：这是发给模型的指令，改译会改变出题行为
        "content": (
            "You are a senior system design interviewer. "
            "Generate exactly one practical, high-quality system design use-case question "
            "tailored to the candidate's background."
        ),
    },
    {
        "role": "user",
        # f-string：把简历与 user_prompt 拼进同一条 user 消息
        "content": f"Resume:\n{resume_text}\n\nPrompt:\n{user_prompt}",
    },
]

# 非流式调用：等整段生成完再取 choices[0].message.content
question_response = openai.chat.completions.create(
    model=MODEL_GPT,
    messages=question_messages,
    # temperature=0.7：略有创造性，适合出题（太低题目会过于死板）
    temperature=0.7,
)

# 取出助手回复文本，并 strip 掉首尾空白
generated_question = question_response.choices[0].message.content.strip()

# 在笔记本里用 Markdown 标题展示生成的题目（展示用英文标题保持原样）
display(Markdown("## Your System Design Question\n\n" + generated_question))


## Generated System Design Question

You are part of a team tasked with migrating Asibiti's core EMR (Electronic Medical Records) database, which is currently using PostgreSQL as its primary data store and operates within an AWS cloud infrastructure environment, to a new decentralized system leveraging blockchain technology. The goal is to enhance security, ensure patient confidentiality compliance with HIPAA regulations in the United States (transposing this requirement into Nigerian contexts like NHIS), and improve interoperability between different healthcare providers across various regions of West Africa while maintaining high data access speed for end-users.

Given that Asibiti's EMR system supports real-time transaction processing, invoicing validation, CBS settlement systems, as well as patient registration trend charts and revenue analytics with observability features:
1. Design a distributed ledger architecture to migrate the existing PostgreSQL database while ensuring zero data loss during transition. Explain how you would use technologies like Docker or Kubernetes for containerization of your services, orchestrate their deployment across multiple AWS regions considering latency and legal jurisdictions (Nigeria's equivalent in this case), as well as the integration with existing APIs such as Flutterwave payments system.
2. Describe how you will maintain data consistency and integrity during real-time processing, addressing potential points of failure within a decentralized environment. Consider using Rust for critical sections due to its performance benefits in concurrent operations – detail your strategy on integrating this language into the existing stack where it would be most beneficial without causing disruinous changes or downtime during migration and daily operations (e.g., payment processing, patient data management).
3. Considering my experience with implementing cursor-based queries for large datasets to improve query execution time by 75% at Asibiti, advise on how you could replicate this performance optimization in the new system architecture while ensuring that blockchain operations do not introduce significant latency into real-time data transactions and access.
4. Outline a plan for continuous compliance monitoring with evolving regulations across different jurisdictions without human intervention - as I have experience automating deployment pipelines using CI/CD practices, suggest how you could extend this to include regulatory changes pertinent to healthcare information privacy laws and blockchain technology use in Nigeria.
5. Lastly, considering the need for a user-friendly interface that enhances patient engagement while providing secure access within Asibiti's dynamic hospital analytics dashboard: Explain how you would approach redesigning this feature to accommodate new data structures introduced by blockchain technology and ensure it remains intuitive for both healthcare providers and patients. Take into account the use of real-time graphical trend charts, leaderboards rankings, as well as logging capabilities that may be affected during migration to a decentralized system architecture built on top of smart contracts or other blockchain primitives where applicable in your design approach while ensuring backend support and scalability.

Please provide high-level thoughts for each part within this use-case question, considering the technical depth required by my background as well as potential challenges I might face during such a complex migration project involving real-time systems across multiple jurisdictions with strict regulatory compliance needs. Your response should reflect both practical knowledge of system design and adaptation to new technologies like blockchain while respecting existing expertise in full stack development, Docker/Kubernetes orchestration, TypeScript proficiency for type safety guarantees within the migration plan, Node.js environments as they relate to real-time systems, Rust integration where needed due to performance considerations and maintainability of codebase during transition phases, MySQL experience when dealing with legacy databases or interfacing between different database technologies post-migration, familiarity with AWS services for cloud infrastructure management within the design context.

## 输入你对题目的作答

把你的系统设计回答写进下一格的 `user_response`，再运行评分单元格。


In [ ]:
# ========== 考生作答：占位字符串，换成你的系统设计答案 ==========

# 变量名 user_response：后面评分格会把它和 generated_question 一起发给模型
user_response = """
<TYPE_YOUR_SYSTEM_DESIGN_ANSWER_HERE>
"""


In [ ]:
# ========== 再调一次 Chat Completions：按固定 Markdown 结构打分反馈 ==========

# grading_messages：system 规定评分量表与输出模板；user 塞题目 + 考生作答
grading_messages = [
    {
        "role": "system",
        # system prompt / 输出模板保持英文：模型按此精确结构生成，翻译会破坏约定格式
        "content": (
            "You are a strict but helpful system design interviewer. "
            "Grade the candidate response on a 10-point scale and provide concise feedback. "
            "Use this exact markdown structure:\n"
            "### Grade\n"
            "<score>/10\n\n"
            "### What went well\n"
            "- ...\n\n"
            "### Gaps\n"
            "- ...\n\n"
            "### Better answer outline\n"
            "- ..."
        ),
    },
    {
        "role": "user",
        "content": (
            # 依赖上一格生成的 generated_question，以及你填写的 user_response
            f"Question:\n{generated_question}\n\n"
            f"Candidate Response:\n{user_response}"
        ),
    },
]

# 同样非流式、temperature=0.7：反馈略有变化但仍围绕模板
grading_response = openai.chat.completions.create(
    model=MODEL_GPT,
    messages=grading_messages,
    temperature=0.7,
)

# 取出评分正文
grading_feedback = grading_response.choices[0].message.content.strip()

# Markdown 展示完整面试反馈
display(Markdown("## Interview Feedback\n\n" + grading_feedback))
